In [1]:
import os
import pystac_client
import requests
import glob
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
from tqdm.notebook import tqdm

In [2]:
# Configurações do Catálogo BDC
servico = "https://data.inpe.br/bdc/stac/v1/"
catalogo = pystac_client.Client.open(servico)
bbox = (-49.87930174, -20.37185787, -49.53239264, -20.03585787)
date_range = "2024-08-01/2024-08-31" 
bandas_desejadas = ["B04", "B08", "B12", "SCL"]

In [3]:
search = catalogo.search(
    bbox=bbox,
    collections=["S2_L2A-1"],
    datetime=date_range
)

In [4]:
items = list(search.get_all_items())
print(f"Encontrados {len(items)} itens. Iniciando download de {len(bandas_desejadas)} bandas por item.")

data_dir = "./data_multi_band_avg"
os.makedirs(data_dir, exist_ok=True)

# 2. Loop de download automatizado
for item in tqdm(items, desc="Progresso por Cena"):
    for band in bandas_desejadas:
        url = item.assets[band].href
        filename = f"{item.id}_{band}.tif"
        filepath = os.path.join(data_dir, filename)
        
        if not os.path.exists(filepath):
            resp = requests.get(url)
            with open(filepath, "wb") as f:
                f.write(resp.content)

Encontrados 12 itens. Iniciando download de 4 bandas por item.


/home/matheus-sales/miniconda3/lib/python3.13/site-packages/pystac_client/item_search.py:940: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


Progresso por Cena:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
valores_validos = [4, 5, 6]

# Processar cada banda espectral (exceto a SCL)
for banda in [b for b in bandas_desejadas if b != "SCL"]:
    print(f"Processando Média Temporal para a banda: {banda}")
    
    arquivos_espectrais = sorted(glob.glob(os.path.join(data_dir, f"*_{banda}.tif")))
    arquivos_scl = sorted(glob.glob(os.path.join(data_dir, "*_SCL.tif")))
    
    pilha_banda = []
    meta = None

    for f_esp, f_scl in zip(arquivos_espectrais, arquivos_scl):
        with rasterio.open(f_esp) as src_esp, rasterio.open(f_scl) as src_scl:
            if meta is None:
                meta = src_esp.profile
                meta.update(dtype=rasterio.float32, nodata=np.nan)

            data = src_esp.read(1).astype(np.float32)
            
            # Reamostragem da SCL para bater com a resolução da banda alvo
            mask = src_scl.read(
                1, 
                out_shape=(src_esp.height, src_esp.width),
                resampling=Resampling.nearest
            )

            # Aplicação da máscara de nuvens (Parâmetro obrigatório)
            data[~np.isin(mask, valores_validos)] = np.nan
            data[data == 0] = np.nan
            
            pilha_banda.append(data)

    # Cálculo da média por pixel ignorando NaNs
    cubo = np.stack(pilha_banda)
    media_banda = np.nanmean(cubo, axis=0)

    # Gravação do resultado sintético
    output_path = f"resultado_avg_{banda}.tif"
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(media_banda, 1)
    
    print(f"Ficheiro {output_path} gerado com sucesso.")